# Week 3: Unsupervised Learning and Clustering Analysis

Dataset: Palmer Penguins CSV from seaborn-data. Run all cells in order. Use actual outputs in the report. Clustering is exploratory and does not prove species labels.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
sns.set_theme(style="whitegrid")
Path("figures").mkdir(exist_ok=True)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url)
print("Shape:", df.shape)
display(df.head())
display(df.describe(include="all").T)
display(df.isna().sum().to_frame("missing_count"))

## Feature selection and preprocessing
Use four numeric measurements. Species and island are excluded from model inputs. Median imputation handles missing numeric values; standardization ensures variables with larger units do not dominate distance calculations.

In [ ]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X_raw = df[features].copy()
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
print("Rows:", X_scaled.shape[0], "| Features:", features)
print("Missing values after imputation:", int(np.isnan(X_scaled).sum()))

## Select cluster count
Use inertia (elbow method) and silhouette score as diagnostics. Neither metric alone proves the correct number of clusters.

In [ ]:
k_values = range(2, 8)
inertias, silhouettes = [], []
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
metrics = pd.DataFrame({"k": list(k_values), "inertia": inertias, "silhouette_score": silhouettes})
display(metrics.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(metrics.k, metrics.inertia, marker="o")
axes[0].set(title="K-Means Elbow Analysis", xlabel="Number of clusters (k)", ylabel="Inertia")
axes[1].plot(metrics.k, metrics.silhouette_score, marker="o")
axes[1].set(title="Silhouette Score by k", xlabel="Number of clusters (k)", ylabel="Silhouette score")
plt.tight_layout()
plt.savefig("figures/01_cluster_selection.png", dpi=160, bbox_inches="tight")
plt.show()

## Fit K-Means
k=3 is a reproducible starting point. Review the metrics and explain whether this choice is supported; adjust if necessary.

In [ ]:
chosen_k = 3
km = KMeans(n_clusters=chosen_k, random_state=42, n_init=10)
df_clustered = df.copy()
df_clustered["cluster"] = km.fit_predict(X_scaled)
print("Cluster sizes:")
display(df_clustered["cluster"].value_counts().sort_index().to_frame("count"))
print("K-Means silhouette:", round(silhouette_score(X_scaled, df_clustered.cluster), 3))

## Visualize clusters with PCA
PCA projects the four standardized inputs into two dimensions for plotting; clustering still uses all four features.

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X_scaled)
plot_df = pd.DataFrame({"PC1": coords[:,0], "PC2": coords[:,1], "cluster": df_clustered.cluster.astype(str)})
plt.figure(figsize=(8,6))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="cluster", palette="tab10", s=55, alpha=.8)
plt.title("K-Means Clusters in PCA Projection")
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(title="Cluster")
plt.tight_layout()
plt.savefig("figures/02_kmeans_pca_clusters.png", dpi=160, bbox_inches="tight")
plt.show()
print("Variance explained by 2 PCs:", round(pca.explained_variance_ratio_.sum()*100, 2), "%")

## Cluster profiles
Compare average measurements per cluster. Species labels are not used to fit the clustering; the optional crosstab is only a post-hoc descriptive check.

In [ ]:
profile = df_clustered.groupby("cluster")[features].mean().round(2)
profile["number_of_rows"] = df_clustered.groupby("cluster").size()
display(profile)
if "species" in df_clustered:
    display(pd.crosstab(df_clustered.cluster, df_clustered.species, normalize="index").round(3))

## Compare with hierarchical clustering
Agglomerative clustering with Ward linkage is compared using the same scaled features and k.

In [ ]:
agg = AgglomerativeClustering(n_clusters=chosen_k, linkage="ward")
agg_labels = agg.fit_predict(X_scaled)
comparison = pd.DataFrame({
    "algorithm": ["K-Means", "Agglomerative (Ward)"],
    "n_clusters": [chosen_k, chosen_k],
    "silhouette_score": [silhouette_score(X_scaled, df_clustered.cluster), silhouette_score(X_scaled, agg_labels)]
})
display(comparison.round(3))
df_clustered["hierarchical_cluster"] = agg_labels
df_clustered.to_csv("penguins_with_clusters.csv", index=False)
profile.to_csv("cluster_profiles.csv")
metrics.to_csv("cluster_selection_metrics.csv", index=False)
comparison.to_csv("algorithm_comparison.csv", index=False)
print("Exported clustered data and summary CSV files.")

## Report reflection prompts
- Which k values scored well on silhouette, and what did the elbow curve suggest?
- How do mean bill length, bill depth, flipper length, and body mass differ by cluster?
- Does the PCA plot show clear separation or overlap?
- How do K-Means and hierarchical clustering compare?
- What are the limitations of interpreting clusters as biological groups?

Source: https://github.com/mwaskom/seaborn-data/blob/master/penguins.csv

Do not claim clusters are species labels. Species was not used as a model input; any cross-tab is only descriptive.